# Lab Assignment 3 — Task 2

## Converting a CFG to Chomsky Normal Form (CNF)

Many parsing algorithms, including CKY, require the grammar to be in **Chomsky Normal Form (CNF)**.

A grammar is in CNF if its production rules have one of these forms:

1. `A -> B C` — a non-terminal produces two non-terminals.
2. `A -> 'word'` — a non-terminal produces a single terminal.

## PART 1 : The Challenge of Ambiguity & CNF Constraints

Consider:

> **I saw the man with a telescope.**

This sentence has two possible interpretations:

1. I saw a man who was holding a telescope.
2. I used a telescope to see the man.

A PCFG assigns probabilities to grammar rules. The probability of a complete parse tree is obtained by multiplying the probabilities of all production rules used in that tree.

### Your task

1. **Convert the given PCFG into strict CNF.** Identify the rule that violates CNF and replace it with equivalent binary rule(s). Preserve the probability of the original alternative.
2. **Parse the sentence using the CNF PCFG and `ViterbiParser`.** Print the most probable parse tree and its total probability.
3. **Explain mathematically why the parser selected this interpretation over the alternative.** Show the rule probabilities used in the selected tree and compare the resulting probability with the competing interpretation.

In [ ]:
pcfg_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V NP PP [0.3]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

sentence = "I saw the man with a telescope"

In [ ]:
import nltk
from nltk import PCFG
from nltk.parse import ViterbiParser

# Convert the non-CNF rule:
#     VP -> V NP PP [0.3]
# into binary rules. Choose an intermediate non-terminal and
# distribute the probability so that the original derivation
# keeps the same probability.

# Example:
#     VP -> V VP_PP [0.3]
#     VP_PP -> NP PP [1.0]

cnf_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V VP_PP [0.3]
    VP_PP -> NP PP [1.0]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

### Analysis requirement

Do not hard-code the final probability.

Use the parse tree returned by the parser to determine which rules were selected. Then calculate the probability of the competing interpretation from the corresponding rule probabilities.

Your explanation should make clear that **ViterbiParser selects the parse with the highest product of rule probabilities**.

# Part 2: Advanced Dependency Parsing Using spaCy

Use the **spaCy NLP library** to perform dependency parsing and graph traversal on the following sentence:

> **The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation.**


## 1. Initialize and Parse

Load the `en_core_web_sm` model in spaCy and process the sentence.

Return the parsed object and store it as **`parsed_doc`**.

## 2. Isolate the Root

Create a function that accepts **`parsed_doc` as its only parameter**.

Iterate through the document to find the structural center of the sentence, where the dependency tag is `"ROOT"`.

Return this token and save it as **`root_node`**.


## 3. Extract the Graph Terminals

Write a function that requires **both `root_node` and `parsed_doc`** as inputs.

1. Check the `.children` of `root_node` to find the nominal subject (`nsubj` dependency), isolating **`researchers`** as `start_node`.
2. Find the token whose text is **`mutation`** in `parsed_doc`, isolating it as `end_node`.
3. Return both nodes.

## 4. Graph Traversal — Shortest Path

Treat the dependency parse as an **undirected graph**. From any token, you may move to:

- its `.head`, or
- any of its `.children`.

Write a **Breadth-First Search (BFS)** function that takes `start_node` and `end_node` as inputs.

Find the shortest path between the two words and return it as a list of tokens named **`dependency_path`**.

## 5. Path Analysis

Write a function that accepts **`dependency_path`** as its input.

Iterate through **this list of tokens only**, not the whole sentence, and print their syntactic details in the following format:

| Word | POS | Head | Dependency |
|---|---|---|---|
| researchers | NOUN | discovered | nsubj |
| ... | ... | ... | ... |

This should extract only the most important syntactic skeleton connecting the two concepts.

## 6. Visual Verification

Pass the original **`parsed_doc`** into spaCy's `displacy` module to render the full dependency tree.

Use the resulting diagram to visually verify that the table generated in Step 5 matches the shortest path in the dependency tree.